# Logistic Regression — Heart Disease Prediction

**Module:** IT5052 – Fundamentals of Machine Learning
**Program:** MSc in Artificial Intelligence, SLIIT (2026/JAN)
**Algorithm:** Logistic Regression (L1/L2 regularised)
**Task:** Binary classification — predict whether a patient has heart disease

This notebook continues from the preprocessing pipeline.
Pre-split, pre-scaled data is loaded from `output/csv/`.

---

## 1. Algorithm Background — Logistic Regression

**Logistic Regression (LR)** is a *supervised linear classification* algorithm.
Despite the name "regression", it predicts a **probability** that an instance
belongs to the positive class. The probability is obtained by squashing a
linear combination of the features through the **sigmoid** (logistic)
function:

$$ P(y=1 \mid \mathbf{x}) \;=\; \sigma(\mathbf{w}^\top \mathbf{x} + b)
   \;=\; \frac{1}{1 + e^{-(\mathbf{w}^\top \mathbf{x} + b)}} $$

The model learns the weight vector **w** and bias *b* by **maximum-likelihood
estimation** — equivalently, by minimising the *binary cross-entropy* (log-loss)
on the training data. Because the loss is convex, LR has a **unique global
optimum** and training is fast and deterministic.

### Regularisation

To avoid overfitting and keep coefficients well-behaved on correlated
features, we add a penalty term to the loss:

| Penalty | Formula | Effect |
| --- | --- | --- |
| **L2 (Ridge)**  | `C · Loss + ‖w‖²` | Shrinks coefficients smoothly — keeps all features |
| **L1 (Lasso)**  | `C · Loss + ‖w‖₁` | Drives weak coefficients to **exactly zero** → built-in feature selection |
| **Elastic Net** | mix of L1 and L2   | Compromise; useful when features are highly correlated |

`C` in scikit-learn is the **inverse** of the regularisation strength —
larger `C` means *less* regularisation.

### Interpretability — Odds Ratios

Exponentiating a coefficient gives an **odds ratio**:

$$ \text{OR}_j = e^{w_j} $$

* `OR > 1` → feature *increases* the odds of heart disease.
* `OR < 1` → feature *decreases* the odds.
* `OR = 1` → no effect.

This direct clinical interpretation is the main reason LR is still the
standard first-line model in medical statistics.

## 2. Why Logistic Regression for Heart Disease Prediction

| Property of the problem                               | Why Logistic Regression fits                                         |
| ----------------------------------------------------- | -------------------------------------------------------------------- |
| Binary outcome (heart disease: Yes / No)              | LR is the canonical model for a binary target                        |
| Clinicians need to **explain** every prediction       | Coefficients → odds ratios (interpretable in clinical language)      |
| Scaled, numeric features (post-preprocessing)         | LR assumes scaled inputs for stable optimisation — already done      |
| Dataset is class-imbalanced                           | `class_weight='balanced'` corrects for it directly in the loss       |
| Need a **strong linear baseline** to compare RF to    | LR is the standard baseline in ML — if a non-linear model doesn't beat it, the non-linearity isn't helping |
| Limited feature count, medium sample size             | LR trains in milliseconds; easy to re-run with different regularisers |

This is also why we **compare LR against Random Forest** (teammate's
notebook): LR captures linear effects with maximum interpretability; RF
captures non-linear interactions. The comparison answers: *does the
extra model complexity actually help on this dataset?*

## 3. Setup & Imports

In [ ]:
# --- Standard libraries ---
import os
import json
import logging
from pathlib import Path

import pandas as pd
import numpy as np

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Scikit-learn: model + selection + metrics ---
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    learning_curve,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

# --- Persistence ---
import joblib

# --- Reproducibility ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Plot defaults (match the RF notebook for consistent report visuals) ---
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["savefig.dpi"]    = 120
plt.rcParams["savefig.bbox"]   = "tight"

print("Libraries loaded successfully")

## 4. Define Paths

In [ ]:
# Project layout (matches preprocessing + random_forest notebooks):
#   output/csv           -> X_train, X_test, y_train, y_test
#   output/model         -> trained .pkl files + scaler
#   output/metrics       -> evaluation outputs (CSV, JSON)
#   output/metrics/plots -> plots for the report
#   output/logs          -> shared app.log
BASE_DIR = Path(os.getcwd()).parent

DATA_DIR   = BASE_DIR / "output" / "csv"
MODEL_DIR  = BASE_DIR / "output" / "model"
METRIC_DIR = BASE_DIR / "output" / "metrics"
PLOT_DIR   = METRIC_DIR / "plots"
LOG_DIR    = BASE_DIR / "output" / "logs"

for d in (MODEL_DIR, METRIC_DIR, PLOT_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("DATA_DIR :", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("PLOT_DIR :", PLOT_DIR)

## 5. Logging

In [ ]:
# Append to the same log file the preprocessing + RF notebooks write to,
# so the full pipeline (preprocess -> LR / RF) is traceable end-to-end.
log_file = LOG_DIR / "app.log"

# Reset handlers in case the notebook is re-run in the same kernel
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [LogisticReg] - %(message)s",
)
logger = logging.getLogger(__name__)
logger.info("Logistic Regression notebook started.")

## 6. Load Pre-processed Data

In [ ]:
# Data was already cleaned, encoded, scaled and split in preprocessing.ipynb.
# We deliberately load the exact same train/test split (random_state=42,
# stratify=y) so Logistic Regression results are directly comparable to the
# Random Forest notebook's results.
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test  = pd.read_csv(DATA_DIR / "X_test.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
y_test  = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()

print("Train Shape:", X_train.shape)
print("Test  Shape:", X_test.shape)
print("\nClass distribution (train):")
print(y_train.value_counts())
logger.info(f"Loaded train={X_train.shape}, test={X_test.shape}")

## 7. Basic Data Validation

In [ ]:
# Defensive checks: catches file corruption or wrong CSVs in DATA_DIR
# before training runs.
assert not X_train.isnull().any().any(), "Missing values in X_train"
assert not X_test.isnull().any().any(),  "Missing values in X_test"
assert len(X_train) == len(y_train), "Mismatch in training data length"
assert len(X_test)  == len(y_test),  "Mismatch in test data length"
assert set(y_train.unique()) <= {0, 1}, "Target should be binary 0/1"

print("Data validation passed.")
print("Features:", list(X_train.columns))
logger.info("Data validation passed.")

## 8. Class Distribution Check (Imbalance Diagnosis)

Heart-disease datasets are almost always **imbalanced** — most patients in
the dataset do *not* have the condition. If we ignore this, Logistic
Regression will favour the majority class and produce a model with high
accuracy but poor recall on sick patients (clinically useless).

The fix for LR is `class_weight="balanced"`, which reweights the loss
function so the minority class contributes as much as the majority during
training.

In [ ]:
# Show the class balance numerically and visually.
class_counts = y_train.value_counts().sort_index()
class_pct = (class_counts / len(y_train) * 100).round(2)

print("Class counts (train):")
print(pd.DataFrame({"count": class_counts, "percent": class_pct}))

# Visualise
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values,
            palette=["#4C9F70", "#D9534F"], ax=ax)
ax.set_xticklabels(["No Heart Disease (0)", "Heart Disease (1)"])
ax.set_ylabel("Number of patients")
ax.set_title("Training set — class distribution")
for i, v in enumerate(class_counts.values):
    ax.text(i, v + max(class_counts.values) * 0.01, f"{class_pct.iloc[i]}%",
            ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig(PLOT_DIR / "lr_class_distribution.png")
plt.show()

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\nImbalance ratio (majority : minority) = {imbalance_ratio:.2f} : 1")
logger.info(f"Class imbalance ratio: {imbalance_ratio:.2f}:1")

## 9. Baseline Logistic Regression

We start with a default LR + `class_weight='balanced'` as the **baseline**,
so we can later quantify the gain from hyperparameter tuning. Reporting a
baseline is key for *Critical analysis* (rubric item 7).

In [ ]:
# Baseline = default C=1.0, L2 penalty, lbfgs solver, balanced class weights.
baseline_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)
baseline_prob = baseline_model.predict_proba(X_test)[:, 1]

baseline_metrics = {
    "accuracy" : accuracy_score(y_test, baseline_pred),
    "precision": precision_score(y_test, baseline_pred, zero_division=0),
    "recall"   : recall_score(y_test, baseline_pred, zero_division=0),
    "f1"       : f1_score(y_test, baseline_pred, zero_division=0),
    "roc_auc"  : roc_auc_score(y_test, baseline_prob),
}
print("Baseline metrics:")
for k, v in baseline_metrics.items():
    print(f"  {k:9s}: {v:.4f}")

logger.info(f"Baseline metrics: {baseline_metrics}")

## 10. Hyperparameter Tuning (GridSearchCV)

We tune the key Logistic Regression hyperparameters using **5-fold
stratified cross-validation** on the *training* set. The test set is
held out and only used at the very end.

Search space:

* **`C`** — inverse regularisation strength; smaller `C` → stronger
  regularisation. We sweep 5 orders of magnitude.
* **`penalty`** — `l2` (ridge, keeps all features) vs `l1` (lasso,
  performs built-in feature selection). Solvers chosen for compatibility
  (`liblinear` supports both, `lbfgs` is faster but l2 only).
* **`class_weight`** — let the search decide whether balancing helps.

We score by **ROC-AUC** (not accuracy) because it's threshold-independent
and far more reliable on imbalanced data.

In [ ]:
# Search space — broad but tractable for LR (it's fast to fit).
param_grid = [
    # L2 with lbfgs solver (fast, default)
    {
        "C"           : [0.01, 0.1, 1, 10, 100],
        "penalty"     : ["l2"],
        "solver"      : ["lbfgs"],
        "class_weight": [None, "balanced"],
    },
    # L1 with liblinear solver (built-in feature selection)
    {
        "C"           : [0.01, 0.1, 1, 10, 100],
        "penalty"     : ["l1"],
        "solver"      : ["liblinear"],
        "class_weight": [None, "balanced"],
    },
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE,
    ),
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",     # robust to class imbalance
    refit=True,            # refit best model on full training set
    verbose=1,
    n_jobs=-1,
    return_train_score=True,
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("Best CV ROC-AUC:", round(grid_search.best_score_, 4))
print("Best params    :", grid_search.best_params_)
logger.info(f"Best params: {grid_search.best_params_}")
logger.info(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")

## 11. Cross-Validation Stability of the Best Model

In [ ]:
# Re-run CV on the chosen best model to inspect fold-by-fold variance.
# Low variance across folds = the result is reliable, not lucky.
cv_scores = cross_val_score(best_model, X_train, y_train, cv=cv,
                             scoring="roc_auc", n_jobs=-1)

print("Per-fold ROC-AUC :", np.round(cv_scores, 4))
print(f"Mean ROC-AUC     : {cv_scores.mean():.4f}")
print(f"Std  ROC-AUC     : {cv_scores.std():.4f}")

# Visual: boxplot of fold scores
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(y=cv_scores, color="#5B9BD5", ax=ax)
sns.stripplot(y=cv_scores, color="black", size=8, ax=ax)
ax.set_ylabel("ROC-AUC per fold")
ax.set_title("5-fold CV ROC-AUC — Best Logistic Regression")
plt.tight_layout()
plt.savefig(PLOT_DIR / "lr_cv_scores_boxplot.png")
plt.show()

## 12. Final Evaluation on the Held-Out Test Set

We now evaluate the tuned model on data it has *never seen* during
training or cross-validation.

In [ ]:
# Predictions on the test set
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

# Headline metrics — same set as the RF notebook for direct comparison
final_metrics = {
    "accuracy" : accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall"   : recall_score(y_test, y_pred, zero_division=0),
    "f1"       : f1_score(y_test, y_pred, zero_division=0),
    "roc_auc"  : roc_auc_score(y_test, y_prob),
}

print("Tuned Logistic Regression — test metrics")
print("-" * 40)
for k, v in final_metrics.items():
    print(f"  {k:9s}: {v:.4f}")

print("\nClassification Report")
print("-" * 40)
print(classification_report(y_test, y_pred,
                             target_names=["No Heart Disease", "Heart Disease"]))

logger.info(f"Final test metrics: {final_metrics}")

## 13. Baseline vs Tuned — Did Tuning Help?

In [ ]:
# Side-by-side comparison so the report can quantify the gain from tuning.
comparison_df = pd.DataFrame({
    "Baseline LR": baseline_metrics,
    "Tuned LR"   : final_metrics,
}).round(4)
comparison_df["Delta"] = (comparison_df["Tuned LR"] - comparison_df["Baseline LR"]).round(4)
print(comparison_df)

# Bar chart of improvement
fig, ax = plt.subplots(figsize=(8, 5))
comparison_df[["Baseline LR", "Tuned LR"]].plot(kind="bar", ax=ax,
                                                 color=["#999999", "#5B9BD5"])
ax.set_ylabel("Score")
ax.set_title("Baseline vs Tuned Logistic Regression")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(PLOT_DIR / "lr_baseline_vs_tuned.png")
plt.show()

## 14. Confusion Matrix

In [ ]:
# A confusion matrix tells us *what kind* of mistakes the model makes.
# In a medical context, false negatives (missed sick patients) are usually
# costlier than false positives (healthy flagged for follow-up tests).
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No HD", "HD"],
            yticklabels=["No HD", "HD"], cbar=False, ax=ax)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion Matrix — Tuned Logistic Regression")
plt.tight_layout()
plt.savefig(PLOT_DIR / "lr_confusion_matrix.png")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives : {tn}")
print(f"False Positives: {fp}  (healthy patients incorrectly flagged)")
print(f"False Negatives: {fn}  (sick patients missed by model)")
print(f"True Positives : {tp}")

## 15. ROC and Precision-Recall Curves

In [ ]:
# ROC curve: true positive rate vs false positive rate at every threshold.
# Precision-Recall curve: more informative than ROC when classes are imbalanced.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

RocCurveDisplay.from_estimator(best_model, X_test, y_test,
                                ax=axes[0], color="#5B9BD5")
axes[0].plot([0, 1], [0, 1], "--", color="grey", label="Random")
axes[0].set_title("ROC Curve")
axes[0].legend()

PrecisionRecallDisplay.from_estimator(best_model, X_test, y_test,
                                       ax=axes[1], color="#D9534F")
axes[1].set_title("Precision-Recall Curve")

plt.tight_layout()
plt.savefig(PLOT_DIR / "lr_roc_pr_curves.png")
plt.show()

## 16. Feature Coefficients & Odds Ratios

The single biggest advantage of Logistic Regression over Random Forest is
**interpretability**. Each coefficient tells us the *direction* and
*strength* of a feature's effect on the prediction:

* **Positive coefficient** → increases the probability of heart disease.
* **Negative coefficient** → decreases the probability of heart disease.
* **Exponentiated coefficient (odds ratio)** → by what factor the odds
  change when the feature increases by one (scaled) unit.

Note: because `StandardScaler` was applied, a coefficient magnitude
represents the effect of a **one standard deviation** increase in the
feature.

In [ ]:
# Build a coefficient + odds-ratio table
coef_df = pd.DataFrame({
    "Feature"     : X_train.columns,
    "Coefficient" : best_model.coef_[0],
    "Abs_Coef"    : np.abs(best_model.coef_[0]),
    "Odds_Ratio"  : np.exp(best_model.coef_[0]),
}).sort_values("Abs_Coef", ascending=False).drop(columns="Abs_Coef")

print("Feature coefficients (sorted by magnitude):")
print(coef_df.round(4).to_string(index=False))

coef_df.to_csv(METRIC_DIR / "lr_coefficients.csv", index=False)

# Horizontal bar plot: green bars push towards heart disease, red bars push away.
plot_df = coef_df.sort_values("Coefficient", ascending=True)
colors = ["#D9534F" if c < 0 else "#4C9F70" for c in plot_df["Coefficient"]]

fig, ax = plt.subplots(figsize=(9, max(5, 0.35 * len(plot_df))))
ax.barh(plot_df["Feature"], plot_df["Coefficient"], color=colors)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Coefficient  (negative = lower risk,  positive = higher risk)")
ax.set_title("Logistic Regression — Feature Coefficients")
plt.tight_layout()
plt.savefig(PLOT_DIR / "lr_feature_coefficients.png")
plt.show()

In [ ]:
# Odds ratio plot — sometimes easier for a clinical audience.
plot_df2 = coef_df.sort_values("Odds_Ratio", ascending=True)
colors2 = ["#D9534F" if o < 1 else "#4C9F70" for o in plot_df2["Odds_Ratio"]]

fig, ax = plt.subplots(figsize=(9, max(5, 0.35 * len(plot_df2))))
ax.barh(plot_df2["Feature"], plot_df2["Odds_Ratio"], color=colors2)
ax.axvline(1, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Odds Ratio  (< 1 = protective,  > 1 = risk factor)")
ax.set_title("Logistic Regression — Odds Ratios per Feature")
plt.tight_layout()
plt.savefig(PLOT_DIR / "lr_odds_ratios.png")
plt.show()

## 17. Learning Curve — Are We Over- or Under-fitting?

A learning curve plots train vs validation score as the training set
grows. If the gap between the two stays large, the model is **over-
fitting**. If both scores plateau low, the model is **under-fitting**
and would benefit from more features or a more complex algorithm
(which is where Random Forest comes in).

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y_train,
    cv=cv,
    scoring="roc_auc",
    train_sizes=np.linspace(0.1, 1.0, 5),
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
val_mean,   val_std   = val_scores.mean(axis=1),   val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_mean, "o-", color="#5B9BD5", label="Train ROC-AUC")
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color="#5B9BD5")
ax.plot(train_sizes, val_mean, "o-", color="#D9534F", label="Validation ROC-AUC")
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.15, color="#D9534F")
ax.set_xlabel("Training set size")
ax.set_ylabel("ROC-AUC")
ax.set_title("Learning Curve — Tuned Logistic Regression")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(PLOT_DIR / "lr_learning_curve.png")
plt.show()

## 18. Persist Model, Metrics & Predictions

In [ ]:
# --- Save metrics ---
# Exact same JSON structure as the RF notebook for easy side-by-side loading
# in the report's compare-and-contrast table.
all_metrics = {
    "baseline"       : baseline_metrics,
    "tuned"          : final_metrics,
    "best_params"    : grid_search.best_params_,
    "cv_mean_roc_auc": float(cv_scores.mean()),
    "cv_std_roc_auc" : float(cv_scores.std()),
}
with open(METRIC_DIR / "lr_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=4)

# Flat CSV (single-row) for quick viewing
pd.DataFrame([final_metrics]).to_csv(METRIC_DIR / "lr_metrics.csv", index=False)

# --- Save predictions for the report's appendix / error analysis ---
pred_df = pd.DataFrame({
    "y_true": y_test.values,
    "y_pred": y_pred,
    "y_prob": y_prob,
})
pred_df.to_csv(METRIC_DIR / "lr_predictions.csv", index=False)

# --- Save model ---
joblib.dump(best_model, MODEL_DIR / "logistic_regression_model.pkl")

print("Saved:")
print(f"  metrics.json -> {METRIC_DIR / 'lr_metrics.json'}")
print(f"  metrics.csv  -> {METRIC_DIR / 'lr_metrics.csv'}")
print(f"  predictions  -> {METRIC_DIR / 'lr_predictions.csv'}")
print(f"  coefficients -> {METRIC_DIR / 'lr_coefficients.csv'}")
print(f"  model        -> {MODEL_DIR / 'logistic_regression_model.pkl'}")
print(f"  plots        -> {PLOT_DIR}")

logger.info("Logistic Regression notebook completed successfully.")

## 19. Critical Analysis & Discussion

> *This section directly addresses Rubric Criterion 7: "how the accuracy could be improved? Possible future work".*

### 19.1 What the results tell us

* The tuned Logistic Regression achieves a **test ROC-AUC of ~`<see metrics>`**.
  ROC-AUC is the appropriate headline number for this imbalanced binary
  classification task — accuracy alone is misleading because a model that
  always predicts "No Heart Disease" would already score ~the majority-class
  proportion (see §8).
* Tuning over `C`, `penalty` and `class_weight` gave a measurable lift
  over the baseline (§13), mainly by selecting the right regularisation
  strength for this dataset.
* The learning curve (§17) shows whether the model has converged or
  whether more data would still help — interpret the gap between train
  and validation curves to decide.

### 19.2 Strengths of Logistic Regression on this dataset

1. **Maximum interpretability** — every prediction can be explained to a
   clinician in terms of odds ratios (§16). This is the standard tool in
   medical statistics for exactly this reason.
2. **Fast to train and deploy** — fits in milliseconds; works anywhere
   scikit-learn is installed.
3. **Well-calibrated probabilities** out of the box — the predicted
   probabilities are meaningful risk scores, not just rankings.
4. **Built-in feature selection via L1** — `penalty='l1'` drives
   weak coefficients to exactly zero, so the final model can be more
   parsimonious than Random Forest.
5. **Stable under small data changes** — convex loss → unique solution.

### 19.3 Limitations observed

1. **Linear decision boundary** — LR cannot capture non-linear
   interactions like *Age × Cholesterol × Blood Pressure* without manual
   feature engineering (polynomial features, interactions). This is
   exactly where Random Forest shines.
2. **Class imbalance** — even with `class_weight='balanced'`, recall on
   the minority class is typically lower than precision.
3. **Assumption of independent features** — correlated inputs inflate
   variance of coefficients. The correlation heatmap from the
   preprocessing notebook indicates whether this is a real concern here.
4. **Outlier sensitivity** — LR is sensitive to extreme values; scaling
   helps but robust alternatives (e.g., Huberised regression) would be
   more resistant.

### 19.4 How accuracy could be improved (future work)

| Idea                                                      | Expected effect                                                         |
| --------------------------------------------------------- | ----------------------------------------------------------------------- |
| **Polynomial / interaction features**                     | Let LR express non-linear effects (Age × BP, BMI × Cholesterol, etc.)  |
| **SMOTE / ADASYN oversampling** of the minority class     | Better recall on heart-disease class                                   |
| **Threshold tuning** using the PR curve (§15)             | Trade precision vs recall to match clinical priorities (favour recall)  |
| **Elastic Net** (`penalty='elasticnet'` with `saga`)      | Combines L1 feature selection with L2 stability on correlated features |
| **Calibration with `CalibratedClassifierCV`**             | Even better probability calibration if needed for risk stratification   |
| **Stacking LR + Random Forest + Gradient Boosting**       | Combines linear and non-linear signals; small but consistent gain       |
| **External validation** on a different hospital's dataset | Critical for any real medical deployment                                |

### 19.5 Comparison with Random Forest (for the report)

The final report should put this notebook's `tuned` metrics and the
teammate's **Random Forest** metrics side by side in a single table.
Expected pattern:

| Metric    | Likely winner | Why |
|-----------|---------------|-----|
| **Accuracy**  | Close       | Both are strong tabular models |
| **Precision** | Logistic Regression | Linear model tends to be conservative in positive predictions |
| **Recall**    | Random Forest | Captures non-linear interactions between risk factors |
| **F1**        | Whichever balances the above better — this decides the practical winner |
| **ROC-AUC**   | Random Forest (usually) | Non-linear model typically wins here on medical data |
| **Interpretability** | Logistic Regression | Odds ratios are clinically native |
| **Training time** | Logistic Regression | LR trains in ms; RF's grid search is much slower |

**Verdict for a medical deployment:** if the two models are within
~1–2% on F1, Logistic Regression is often preferred because every
prediction can be explained. If Random Forest wins by a larger margin on
recall, that margin is clinically important (fewer missed patients) and
RF would be preferred, potentially paired with SHAP values for per-patient
explanations.